In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import torch

RANDOM_SEED = int(os.environ.get('SKIN_RANDOM_SEED', '42'))
CLASS_NAMES = ('AK', 'BCC', 'MEL', 'NV', 'SCC')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def findProjectRoot(startPath: Path) -> Path:
    for candidateRoot in (startPath, *startPath.parents):
        if (candidateRoot / 'data' / 'processed').exists() and (candidateRoot / 'notebooks').exists():
            return candidateRoot
    raise RuntimeError('Run from the skin-domain-shift project or one of its subdirectories.')

projectRoot = findProjectRoot(Path.cwd().resolve())
processedRoot = projectRoot / 'data' / 'processed'
artifactRoot = projectRoot / 'artifacts' / 'ds'
figureRoot = projectRoot / 'reports' / 'figures' / 'ds'
reportRoot = projectRoot / 'reports'
artifactRoot.mkdir(parents=True, exist_ok=True)
figureRoot.mkdir(parents=True, exist_ok=True)

def fromProjectPath(pathValue: str) -> Path:
    return (projectRoot / Path(pathValue)).resolve()

runMode = os.environ.get('SKIN_RUN_MODE', 'full').strip().lower()
if runMode not in {'smoke', 'full'}:
    raise ValueError('SKIN_RUN_MODE must be smoke or full.')
runModeRoot = artifactRoot / runMode
runFigureModeRoot = figureRoot / runMode
runArtifactRoot = runModeRoot / f'seed{RANDOM_SEED}'
runFigureRoot = runFigureModeRoot / f'seed{RANDOM_SEED}'
runArtifactRoot.mkdir(parents=True, exist_ok=True)
runFigureRoot.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {projectRoot}')
print(f'Run mode: {runMode}; seed: {RANDOM_SEED}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Avail')

Project root: D:\sunib\Research\Skin\skin-domain-shift
Run mode: full; seed: 42
Device: cuda
GPU Avail


In [2]:
manifestPaths = {
    'clinicalTrain': processedRoot / 'clinicalTrain.csv',
    'clinicalVal': processedRoot / 'clinicalVal.csv',
    'clinicalTest': processedRoot / 'clinicalTest.csv',
    'isicTest': processedRoot / 'isicTest.csv',
    'exactDuplicates': processedRoot / 'exactDuplicates.csv',
}
for manifestName, manifestPath in manifestPaths.items():
    if not manifestPath.exists():
        raise FileNotFoundError(f'Missing {manifestName}: {manifestPath}. Run Phase 1 first.')

clinicalTrain = pd.read_csv(manifestPaths['clinicalTrain'], low_memory=False)
clinicalVal = pd.read_csv(manifestPaths['clinicalVal'], low_memory=False)
clinicalTest = pd.read_csv(manifestPaths['clinicalTest'], low_memory=False)
isicTest = pd.read_csv(manifestPaths['isicTest'], low_memory=False)
exactDuplicates = pd.read_csv(manifestPaths['exactDuplicates'], low_memory=False)

splitFrames = {
    'clinicalTrain': clinicalTrain,
    'clinicalVal': clinicalVal,
    'clinicalTest': clinicalTest,
    'isicTest': isicTest,
}
for splitName, frame in splitFrames.items():
    missingLabels = set(CLASS_NAMES) - set(frame['harmonizedLabel'].dropna())
    assert not missingLabels, f'{splitName} is missing classes: {sorted(missingLabels)}'
    assert frame['imagePath'].notna().all(), f'{splitName} contains missing image paths.'
    assert frame['imagePath'].map(lambda value: fromProjectPath(value).exists()).all(), f'{splitName} contains missing files.'
    assert not frame['imageId'].duplicated().any(), f'{splitName} contains duplicate image IDs.'

print({name: len(frame) for name, frame in splitFrames.items()})

{'clinicalTrain': 1479, 'clinicalVal': 290, 'clinicalTest': 294, 'isicTest': 5336}


In [3]:
def overlapCount(leftFrame: pd.DataFrame, rightFrame: pd.DataFrame, columnName: str) -> int:
    leftValues = set(leftFrame[columnName].dropna().astype(str))
    rightValues = set(rightFrame[columnName].dropna().astype(str))
    return len(leftValues & rightValues)

clinicalPairs = [
    ('trainVal', clinicalTrain, clinicalVal),
    ('trainTest', clinicalTrain, clinicalTest),
    ('valTest', clinicalVal, clinicalTest),
]
patientOverlaps = {
    pairName: overlapCount(leftFrame, rightFrame, 'patientId')
    for pairName, leftFrame, rightFrame in clinicalPairs
}
duplicateOverlaps = {
    pairName: overlapCount(leftFrame, rightFrame, 'confirmedDuplicateGroup')
    for pairName, leftFrame, rightFrame in clinicalPairs
}
crossDomainExactGroups = int(
    exactDuplicates.loc[exactDuplicates['sourceScope'].eq('crossDataset'), 'duplicateGroup'].nunique()
) if len(exactDuplicates) else 0

assert all(count == 0 for count in patientOverlaps.values()), patientOverlaps
assert all(count == 0 for count in duplicateOverlaps.values()), duplicateOverlaps
assert crossDomainExactGroups == 0, 'Exact duplicates exist across clinical and ISIC domains.'
assert set(clinicalTrain['imageId']).isdisjoint(set(isicTest['imageId']))
assert isicTest['split'].eq('officialTestDomainShift').all()

print(f'Patient overlaps: {patientOverlaps}')
print(f'Confirmed duplicate overlaps: {duplicateOverlaps}')
print(f'Cross-domain exact duplicate groups: {crossDomainExactGroups}')

Patient overlaps: {'trainVal': 0, 'trainTest': 0, 'valTest': 0}
Confirmed duplicate overlaps: {'trainVal': 0, 'trainTest': 0, 'valTest': 0}
Cross-domain exact duplicate groups: 0


In [4]:
classCounts = {
    splitName: frame['harmonizedLabel'].value_counts().reindex(CLASS_NAMES, fill_value=0).astype(int).to_dict()
    for splitName, frame in splitFrames.items()
}
trainCounts = clinicalTrain['harmonizedLabel'].value_counts().reindex(CLASS_NAMES)
classWeights = (len(clinicalTrain) / (len(CLASS_NAMES) * trainCounts)).to_dict()

distributionRows = []
for splitName, counts in classCounts.items():
    for className, imageCount in counts.items():
        distributionRows.append({'split': splitName, 'className': className, 'imageCount': imageCount})
distributionFrame = pd.DataFrame(distributionRows)
display(distributionFrame.pivot(index='className', columns='split', values='imageCount'))
print('Training class weights:', {key: round(value, 4) for key, value in classWeights.items()})

split,clinicalTest,clinicalTrain,clinicalVal,isicTest
className,,,,
AK,103,531,96,374
BCC,124,606,115,975
MEL,7,35,10,1327
NV,38,169,37,2495
SCC,22,138,32,165


Training class weights: {'AK': 0.5571, 'BCC': 0.4881, 'MEL': 8.4514, 'NV': 1.7503, 'SCC': 2.1435}


In [5]:
experimentConfig = {
    'randomSeed': RANDOM_SEED,
    'classNames': list(CLASS_NAMES),
    'modelName': 'mobilenet_v3_small',
    'pretraining': 'ImageNet torchvision default weights',
    'imageSize': 224,
    'selectionMetric': 'clinical validation macroF1',
    'reportedMetrics': ['macroF1', 'balancedAccuracy', 'perClassRecall', 'confusionMatrix'],
    'clinicalSplitCounts': {name: len(frame) for name, frame in splitFrames.items() if name.startswith('clinical')},
    'isicDomainShiftTestCount': len(isicTest),
    'isicUsedForSelection': False,
    'patientOverlaps': patientOverlaps,
    'confirmedDuplicateOverlaps': duplicateOverlaps,
    'crossDomainExactDuplicateGroups': crossDomainExactGroups,
    'classWeights': {key: float(value) for key, value in classWeights.items()},
}
experimentConfigPath = runArtifactRoot / 'experimentConfig.json'
experimentConfigPath.write_text(json.dumps(experimentConfig, indent=2), encoding='utf-8')
distributionPath = runArtifactRoot / 'experimentClassDistribution.csv'
distributionFrame.to_csv(distributionPath, index=False)
print(json.dumps(experimentConfig, indent=2))

{
  "randomSeed": 2026,
  "classNames": [
    "AK",
    "BCC",
    "MEL",
    "NV",
    "SCC"
  ],
  "modelName": "mobilenet_v3_small",
  "pretraining": "ImageNet torchvision default weights",
  "imageSize": 224,
  "selectionMetric": "clinical validation macroF1",
  "reportedMetrics": [
    "macroF1",
    "balancedAccuracy",
    "perClassRecall",
    "confusionMatrix"
  ],
  "clinicalSplitCounts": {
    "clinicalTrain": 1479,
    "clinicalVal": 290,
    "clinicalTest": 294
  },
  "isicDomainShiftTestCount": 5336,
  "isicUsedForSelection": false,
  "patientOverlaps": {
    "trainVal": 0,
    "trainTest": 0,
    "valTest": 0
  },
  "confirmedDuplicateOverlaps": {
    "trainVal": 0,
    "trainTest": 0,
    "valTest": 0
  },
  "crossDomainExactDuplicateGroups": 0,
  "classWeights": {
    "AK": 0.5570621468926553,
    "BCC": 0.4881188118811881,
    "MEL": 8.451428571428572,
    "NV": 1.7502958579881658,
    "SCC": 2.143478260869565
  }
}
